In [ ]:
import os
import math
import json
import random
import copy
import numpy as np
import pandas as pd
from dataclasses import dataclass
from collections import Counter
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# -----------------------------
# 1) Seed
# -----------------------------
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
seed_everything(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# -----------------------------
# 2) Config
# -----------------------------
@dataclass
class CFG:
    ratings_path: str = "/kaggle/input/datasets/cameliabenlaamari/movielens-1m-dataset/ratings.csv"
    min_user_interactions: int = 5
    max_len: int = 200
    mask_prob: float = 0.2
    num_test_negatives: int = 100
    use_all_interactions_as_implicit: bool = True
    # model
    hidden_size: int = 256
    num_layers: int = 2
    num_heads: int = 2
    dropout: float = 0.2
    # training
    batch_size: int = 256
    epochs: int = 300
    lr: float = 1e-4
    weight_decay: float = 0.01
    grad_clip: float = 5.0
    patience: int = 30
    num_workers: int = 2
    warmup_ratio: float = 0.1
    last_mask_prob: float = 0.15         
    # Fine-tune phase
    ft_epochs: int = 30
    ft_lr: float = 1e-5
    ft_patience: int = 10
    # saving
    best_model_path: str = "/kaggle/working/best_bert4rec_ml1m.pt"
    mapping_path: str = "/kaggle/working/item_mapping_bert4rec_ml1m.json"

cfg = CFG()

# -----------------------------
# 3) Load ratings
# -----------------------------
def load_ratings(path):
    seps = [",", ";", "\t", "|"]
    for sep in seps:
        try:
            df = pd.read_csv(path, sep=sep)
            cols = [str(c).strip().replace("\ufeff", "") for c in df.columns]
            df.columns = cols
            if set(["userId", "movieId", "rating", "timestamp"]).issubset(df.columns):
                return df[["userId", "movieId", "rating", "timestamp"]].copy()
            lower_cols = {c.lower(): c for c in df.columns}
            needed = ["userid", "movieid", "rating", "timestamp"]
            if all(c in lower_cols for c in needed):
                df = df.rename(columns={
                    lower_cols["userid"]: "userId",
                    lower_cols["movieid"]: "movieId",
                    lower_cols["rating"]: "rating",
                    lower_cols["timestamp"]: "timestamp",
                })
                return df[["userId", "movieId", "rating", "timestamp"]].copy()
        except Exception:
            continue
    raise ValueError("Could not read ratings file.")

ratings = load_ratings(cfg.ratings_path)
print("Raw ratings:", ratings.shape)


In [ ]:
# -----------------------------
# 4) Preprocess
# -----------------------------
def preprocess_to_sequences(df, cfg):
    if not cfg.use_all_interactions_as_implicit:
        df = df[df["rating"] >= 4].copy()
    df = df.sort_values(["userId", "timestamp", "movieId"]).reset_index(drop=True)
    user_counts = df.groupby("userId")["movieId"].count()
    keep_users = user_counts[user_counts >= cfg.min_user_interactions].index
    df = df[df["userId"].isin(keep_users)].copy()
    user2items = df.groupby("userId")["movieId"].apply(list).to_dict()
    return df, user2items

ratings, user2items_raw = preprocess_to_sequences(ratings, cfg)
print("After preprocess:", ratings.shape)
print("Users:", len(user2items_raw))

# -----------------------------
# 5) Item mapping
# -----------------------------
all_items = sorted(ratings["movieId"].unique().tolist())
item2idx = {item: idx + 1 for idx, item in enumerate(all_items)}
idx2item = {idx: item for item, idx in item2idx.items()}
num_items = len(item2idx)
MASK_TOKEN_ID = num_items + 1
VOCAB_SIZE = num_items + 2

with open(cfg.mapping_path, "w") as f:
    json.dump({
        "item2idx": {str(k): int(v) for k, v in item2idx.items()},
        "idx2item": {str(k): int(v) for k, v in idx2item.items()},
        "mask_token_id": MASK_TOKEN_ID,
    }, f)

user2seq = {u: [item2idx[m] for m in seq if m in item2idx] for u, seq in user2items_raw.items()}
user2seq = {u: seq for u, seq in user2seq.items() if len(seq) >= cfg.min_user_interactions}

# -----------------------------
# 6) Split + Sliding Window Augmentation
# -----------------------------
train_seqs_augmented = []
val_data = []
test_data = []
step_length = 50

for u, seq in user2seq.items():
    val_data.append((u, seq[:-2], seq[-2]))
    test_data.append((u, seq[:-1], seq[-1]))
    train_seq = seq[:-2]
    if len(train_seq) <= cfg.max_len:
        train_seqs_augmented.append(train_seq)
    else:
        idx = len(train_seq)
        while idx > 0:
            start_idx = max(0, idx - cfg.max_len)
            train_seqs_augmented.append(train_seq[start_idx:idx])
            idx -= step_length

print(f"Số lượng users: {len(user2seq)}")
print(f"Số lượng chuỗi huấn luyện (sau augmentation): {len(train_seqs_augmented)}")

# -----------------------------
# 7) Popularity & Fixed Negatives
# -----------------------------
item_counter = Counter()
for seq in train_seqs_augmented:
    item_counter.update(seq)

pop_items = np.array(sorted(item_counter.keys()), dtype=np.int64)
pop_counts = np.array([item_counter[i] for i in pop_items], dtype=np.float64)
pop_probs = pop_counts / pop_counts.sum()
all_item_set = set(range(1, num_items + 1))

def sample_fixed_negatives(user_seen_set, positive_item, num_negatives, rng):
    negatives = set()
    max_trials = num_negatives * 100
    trials = 0
    while len(negatives) < num_negatives and trials < max_trials:
        sampled = rng.choice(pop_items, size=num_negatives * 2, replace=True, p=pop_probs)
        for x in sampled:
            x = int(x)
            if x != positive_item and x not in user_seen_set:
                negatives.add(x)
                if len(negatives) >= num_negatives:
                    break
        trials += 1
    if len(negatives) < num_negatives:
        remain = list(all_item_set - user_seen_set - {positive_item} - negatives)
        rng.shuffle(remain)
        negatives.update(remain[:num_negatives - len(negatives)])
    negatives = list(negatives)[:num_negatives]
    rng.shuffle(negatives)
    return negatives

rng_val = np.random.default_rng(2024)
rng_test = np.random.default_rng(2025)

val_negatives = {}
test_negatives = {}
for u, prefix_seq, target in val_data:
    user_seen = set(prefix_seq)
    val_negatives[u] = sample_fixed_negatives(user_seen, target, cfg.num_test_negatives, rng_val)
for u, prefix_seq, target in test_data:
    user_seen = set(prefix_seq)
    test_negatives[u] = sample_fixed_negatives(user_seen, target, cfg.num_test_negatives, rng_test)

# -----------------------------
# 8) Helpers
# -----------------------------
def left_pad_sequence(seq, max_len, pad_id=0):
    seq = seq[-max_len:]
    if len(seq) < max_len:
        seq = [pad_id] * (max_len - len(seq)) + seq
    return seq

def random_mask_training_instance(seq, mask_prob, mask_token_id):
    tokens = seq.copy()
    labels = [0] * len(tokens)
    candidate_positions = [i for i, t in enumerate(tokens) if t != 0]
    num_to_mask = max(1, int(round(len(candidate_positions) * mask_prob)))
    mask_positions = random.sample(candidate_positions, min(num_to_mask, len(candidate_positions)))
    for pos in mask_positions:
        labels[pos] = tokens[pos]
        tokens[pos] = mask_token_id
    return tokens, labels

def mask_last_item_only(seq, mask_token_id):
    tokens = seq.copy()
    labels = [0] * len(tokens)
    non_zero_indices = [i for i, t in enumerate(tokens) if t != 0]
    if non_zero_indices:
        last_idx = non_zero_indices[-1]
        labels[last_idx] = tokens[last_idx]
        tokens[last_idx] = mask_token_id
    return tokens, labels

In [1]:
# -----------------------------
# 9) Datasets
# -----------------------------
class BERT4RecDataset(Dataset):
    def __init__(self, train_seqs_list, max_len, mask_prob, mask_token_id, last_mask_prob=0.15):
        self.train_seqs = train_seqs_list
        self.max_len = max_len
        self.mask_prob = mask_prob
        self.mask_token_id = mask_token_id
        self.last_mask_prob = last_mask_prob

    def __len__(self):
        return len(self.train_seqs)

    def __getitem__(self, idx):
        seq = self.train_seqs[idx]
        padded = left_pad_sequence(seq, self.max_len, pad_id=0)
        if random.random() < self.last_mask_prob:
            tokens, labels = mask_last_item_only(padded, self.mask_token_id)
        else:
            tokens, labels = random_mask_training_instance(padded, self.mask_prob, self.mask_token_id)
        return torch.tensor(tokens, dtype=torch.long), torch.tensor(labels, dtype=torch.long)

class BERT4RecFinetuneDataset(Dataset):
    def __init__(self, train_seqs_list, max_len, mask_token_id):
        self.train_seqs = train_seqs_list
        self.max_len = max_len
        self.mask_token_id = mask_token_id

    def __len__(self):
        return len(self.train_seqs)

    def __getitem__(self, idx):
        seq = self.train_seqs[idx]
        padded = left_pad_sequence(seq, self.max_len, pad_id=0)
        tokens, labels = mask_last_item_only(padded, self.mask_token_id)
        return torch.tensor(tokens, dtype=torch.long), torch.tensor(labels, dtype=torch.long)

class BERT4RecEvalDataset(Dataset):
    def __init__(self, data, negatives_dict, max_len, mask_token_id):
        self.data = data
        self.negatives_dict = negatives_dict
        self.max_len = max_len
        self.mask_token_id = mask_token_id

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        u, prefix_seq, target = self.data[idx]
        seq = left_pad_sequence(prefix_seq[-(self.max_len - 1):] + [self.mask_token_id], self.max_len, pad_id=0)
        negatives = self.negatives_dict[u]
        candidates = [target] + negatives
        return (
            torch.tensor(seq, dtype=torch.long),
            torch.tensor(target, dtype=torch.long),
            torch.tensor(u, dtype=torch.long),
            torch.tensor(candidates, dtype=torch.long),
        )

def eval_collate_fn(batch):
    seqs, targets, users, candidates = zip(*batch)
    return torch.stack(seqs), torch.stack(targets), torch.stack(users), torch.stack(candidates)

# -----------------------------
# 10) DataLoader
# -----------------------------
train_loader = DataLoader(
    BERT4RecDataset(train_seqs_augmented, cfg.max_len, cfg.mask_prob, MASK_TOKEN_ID, cfg.last_mask_prob),
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=True,
)

finetune_loader = DataLoader(
    BERT4RecFinetuneDataset(train_seqs_augmented, cfg.max_len, MASK_TOKEN_ID),
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=True,
)

val_loader = DataLoader(
    BERT4RecEvalDataset(val_data, val_negatives, cfg.max_len, MASK_TOKEN_ID),
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=True,
    collate_fn=eval_collate_fn,
)

test_loader = DataLoader(
    BERT4RecEvalDataset(test_data, test_negatives, cfg.max_len, MASK_TOKEN_ID),
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=True,
    collate_fn=eval_collate_fn,
)

# -----------------------------
# 11) Model (pure post-LN)
# -----------------------------
class BERT4Rec(nn.Module):
    def __init__(self, vocab_size, max_len, hidden_size=256, num_layers=2, num_heads=2, dropout=0.2):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.max_len = max_len
        self.item_embedding = nn.Embedding(vocab_size, hidden_size, padding_idx=0)
        self.position_embedding = nn.Embedding(max_len, hidden_size)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size,
            nhead=num_heads,
            dim_feedforward=hidden_size * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=False,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.out_proj = nn.Linear(hidden_size, hidden_size)
        self.out_gelu = nn.GELU()
        self.out_bias = nn.Parameter(torch.zeros(vocab_size))
        self._reset_parameters()

    def _reset_parameters(self):
        nn.init.trunc_normal_(self.item_embedding.weight, mean=0.0, std=0.02, a=-0.04, b=0.04)
        nn.init.trunc_normal_(self.position_embedding.weight, mean=0.0, std=0.02, a=-0.04, b=0.04)
        nn.init.trunc_normal_(self.out_proj.weight, mean=0.0, std=0.02, a=-0.04, b=0.04)
        nn.init.zeros_(self.out_proj.bias)
        with torch.no_grad():
            self.item_embedding.weight[0].fill_(0.0)

    def _build_position_ids(self, seq):
        non_pad = (seq != 0).long()
        pos_ids = torch.cumsum(non_pad, dim=1) - 1
        pos_ids = pos_ids.clamp(min=0)
        return pos_ids

    def forward(self, seq):
        pos_ids = self._build_position_ids(seq)
        x = self.item_embedding(seq) + self.position_embedding(pos_ids)
        padding_mask = (seq == 0)
        h = self.encoder(x, src_key_padding_mask=padding_mask)
        h_proj = self.out_gelu(self.out_proj(h))
        logits = (h_proj @ self.item_embedding.weight.T) + self.out_bias
        return logits

model = BERT4Rec(VOCAB_SIZE, cfg.max_len, cfg.hidden_size, cfg.num_layers, cfg.num_heads, cfg.dropout).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, betas=(0.9, 0.999), weight_decay=cfg.weight_decay)

total_steps = len(train_loader) * cfg.epochs
warmup_steps = int(total_steps * cfg.warmup_ratio)

def lr_lambda(current_step):
    if current_step < warmup_steps:
        return float(current_step) / float(max(1, warmup_steps))
    return max(0.0, float(total_steps - current_step) / float(max(1, total_steps - warmup_steps)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

# -----------------------------
# 12) Metrics
# -----------------------------
@torch.no_grad()
def evaluate(model, loader, topk=10):
    model.eval()
    total_hr, total_ndcg, total_mrr, total_count = 0.0, 0.0, 0.0, 0
    for seq, target, users, candidates in loader:
        seq = seq.to(DEVICE)
        candidates = candidates.to(DEVICE)
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            last_logits = model(seq)[:, -1, :]
        last_logits[:, 0] = -1e9
        last_logits[:, MASK_TOKEN_ID] = -1e9
        cand_scores = torch.gather(last_logits, 1, candidates)
        ranks = torch.argsort(torch.argsort(-cand_scores, dim=1), dim=1)[:, 0] + 1
        ranks = ranks.cpu().numpy()
        for rank in ranks:
            if rank <= topk:
                total_hr += 1.0
                total_ndcg += 1.0 / math.log2(rank + 1)
                total_mrr += 1.0 / rank
            total_count += 1
    return {
        "HR@10": total_hr / total_count,
        "NDCG@10": total_ndcg / total_count,
        "MRR": total_mrr / total_count,
    }

def is_better(curr, best):
    if best is None:
        return True
    if curr["NDCG@10"] != best["NDCG@10"]:
        return curr["NDCG@10"] > best["NDCG@10"]
    if curr["HR@10"] != best["HR@10"]:
        return curr["HR@10"] > best["HR@10"]
    return curr["MRR"] > best["MRR"]

# -----------------------------
# 13) MAIN TRAINING
# -----------------------------
best_metrics = None
best_epoch = -1
best_state = None
patience_counter = 0

for epoch in range(1, cfg.epochs + 1):
    model.train()
    total_loss, total_tokens = 0.0, 0
    for seq, labels in train_loader:
        seq = seq.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            logits = model(seq)
            loss = criterion(logits.view(-1, VOCAB_SIZE), labels.view(-1))
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        num_masked = max(1, (labels != 0).sum().item())
        total_loss += loss.item() * num_masked
        total_tokens += num_masked

    val_metrics = evaluate(model, val_loader, topk=10)
    print(f"Epoch {epoch:03d} | Train Loss {total_loss / total_tokens:.4f} | "
          f"Val HR@10 {val_metrics['HR@10']:.4f} | Val NDCG@10 {val_metrics['NDCG@10']:.4f} | Val MRR {val_metrics['MRR']:.4f}")

    if is_better(val_metrics, best_metrics):
        best_metrics = val_metrics.copy()
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "config": cfg.__dict__,
            "vocab_size": VOCAB_SIZE,
            "mask_token_id": MASK_TOKEN_ID,
            "best_epoch": best_epoch,
            "best_metrics": best_metrics,
        }, cfg.best_model_path)
    else:
        patience_counter += 1
        if patience_counter >= cfg.patience:
            print(f"Early stopping at epoch {epoch}")
            break

# -----------------------------
# 14) FINE-TUNE PHASE (CHỈ MASK LAST ITEM)
# -----------------------------
print("\n" + "="*60)
print("BẮT ĐẦU FINE-TUNING PHASE (CHỈ MASK LAST ITEM - lr=1e-5)")
print("="*60)

model.load_state_dict(best_state)

ft_optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.ft_lr,
    betas=(0.9, 0.999),
    weight_decay=cfg.weight_decay,
)

ft_best_metrics = best_metrics.copy()
ft_best_epoch = best_epoch
ft_patience_counter = 0

for epoch in range(1, cfg.ft_epochs + 1):
    model.train()
    total_loss, total_tokens = 0.0, 0
    for seq, labels in finetune_loader:
        seq = seq.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        ft_optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            logits = model(seq)
            loss = criterion(logits.view(-1, VOCAB_SIZE), labels.view(-1))
        scaler.scale(loss).backward()
        scaler.unscale_(ft_optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(ft_optimizer)
        scaler.update()
        num_masked = max(1, (labels != 0).sum().item())
        total_loss += loss.item() * num_masked
        total_tokens += num_masked

    val_metrics = evaluate(model, val_loader, topk=10)
    print(f"FT Epoch {epoch:03d} | Loss {total_loss/total_tokens:.4f} | "
          f"Val HR@10 {val_metrics['HR@10']:.4f} | Val NDCG@10 {val_metrics['NDCG@10']:.4f} | Val MRR {val_metrics['MRR']:.4f}")

    if is_better(val_metrics, ft_best_metrics):
        ft_best_metrics = val_metrics.copy()
        ft_best_epoch = epoch
        ft_patience_counter = 0
        best_state = copy.deepcopy(model.state_dict())
        torch.save({
            "model_state_dict": model.state_dict(),
            "config": cfg.__dict__,
            "vocab_size": VOCAB_SIZE,
            "mask_token_id": MASK_TOKEN_ID,
            "best_epoch": ft_best_epoch,
            "best_metrics": ft_best_metrics,
        }, cfg.best_model_path)
    else:
        ft_patience_counter += 1
        if ft_patience_counter >= cfg.ft_patience:
            print(f"Fine-tuning early stopping at epoch {epoch}")
            break

# -----------------------------
# 15) FINAL TEST SAU FINE-TUNE
# -----------------------------
model.load_state_dict(best_state)
test_metrics = evaluate(model, test_loader, topk=10)
print("\n===== FINAL TEST SAU FINE-TUNE =====")
print(f"Best epoch          : {ft_best_epoch}")
print(f"Test HR@10          : {test_metrics['HR@10']:.4f}")
print(f"Test NDCG@10        : {test_metrics['NDCG@10']:.4f}")
print(f"Test MRR            : {test_metrics['MRR']:.4f}")
print(f"Model saved at      : {cfg.best_model_path}")

Device: cuda
Raw ratings: (1000209, 4)
After preprocess: (1000209, 4)
Users: 6040
Số lượng users: 6040
Số lượng chuỗi huấn luyện (sau augmentation): 18263
Epoch 001 | Train Loss 8.2178 | Val HR@10 0.1002 | Val NDCG@10 0.0457 | Val MRR 0.0296
Epoch 002 | Train Loss 8.2109 | Val HR@10 0.1038 | Val NDCG@10 0.0485 | Val MRR 0.0320
Epoch 003 | Train Loss 8.1833 | Val HR@10 0.1118 | Val NDCG@10 0.0508 | Val MRR 0.0327
Epoch 004 | Train Loss 8.0741 | Val HR@10 0.1179 | Val NDCG@10 0.0544 | Val MRR 0.0355
Epoch 005 | Train Loss 7.8406 | Val HR@10 0.1273 | Val NDCG@10 0.0607 | Val MRR 0.0407
Epoch 006 | Train Loss 7.6322 | Val HR@10 0.1349 | Val NDCG@10 0.0667 | Val MRR 0.0462
Epoch 007 | Train Loss 7.5662 | Val HR@10 0.1366 | Val NDCG@10 0.0672 | Val MRR 0.0465
Epoch 008 | Train Loss 7.5505 | Val HR@10 0.1382 | Val NDCG@10 0.0676 | Val MRR 0.0465
Epoch 009 | Train Loss 7.5447 | Val HR@10 0.1364 | Val NDCG@10 0.0670 | Val MRR 0.0463
Epoch 010 | Train Loss 7.5287 | Val HR@10 0.1368 | Val NDCG@10